# Ransomware Detection Using Machine Learning

**Course No:** CSE-312  
**Course Title:** Computer Networks (Sessional)  
**Lab Report Title:** Ransomware Detection Using Machine Learning  

---

## Introduction
The relentless evolution of ransomware poses a critical threat to digital infrastructure worldwide. Traditional signature-based defenses are increasingly inadequate against novel ransomware variants. Machine learning provides a data-driven alternative — learning behavioral patterns from endpoint telemetry to detect ransomware activity in real time. This lab report applies several machine learning algorithms to a ransomware behavior dataset and compares their detection performance.

## Objectives
- To understand various Machine Learning algorithms
- To apply them and perform ransomware detection on the dataset
- To compare the accuracy of various machine learning algorithms

## Description
This report evaluates K-Nearest Neighbors (KNN), Random Forest, Decision Tree, Naive Bayes, Logistic Regression, and SVM on a ransomware behavior telemetry dataset. Features include file encryption rate, entropy delta, shadow copy deletion attempts, process spawn rate, and more. Each model is trained, validated, and assessed with classification reports and confusion matrices.

## Code Analysis
### 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

plt.style.use('ggplot')
print('Libraries imported successfully.')

### 2. Generate Synthetic Ransomware Dataset

In [ ]:
rng = np.random.default_rng(42)
N = 5000
label = rng.binomial(1, 0.32, N)

data = pd.DataFrame({
    'file_modifications_per_min': rng.poisson(35 + label * 180),
    'files_encrypted_per_min':    rng.poisson(1  + label * 95),
    'entropy_delta':              np.clip(rng.normal(0.35 + label * 1.85, 0.35), 0, None),
    'ransom_note_created':        rng.binomial(1, 0.03 + label * 0.74),
    'shadow_copy_delete_attempt': rng.binomial(1, 0.02 + label * 0.68),
    'backup_service_stop':        rng.binomial(1, 0.01 + label * 0.58),
    'suspicious_ext_writes':      rng.poisson(1  + label * 25),
    'process_spawn_rate':         rng.poisson(5  + label * 28),
    'high_privilege_token_use':   rng.binomial(1, 0.09 + label * 0.51),
    'outbound_unique_ips':        rng.poisson(2  + label * 8),
    'cpu_usage_percent':          np.clip(rng.normal(22 + label * 41, 10), 0, 100),
    'disk_write_mb_s':            np.clip(rng.normal(7  + label * 35, 8),  0, None),
    'is_ransomware':              label
})

print(data.shape)
data.head()

### 3. Dataset Info

In [ ]:
data.info()

### 4. Class Distribution

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='is_ransomware', data=data, palette='Set2')
ax.set_xticklabels(['Benign', 'Ransomware'])
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### 5. Train / Test Split

In [ ]:
X = data.drop('is_ransomware', axis=1)
y = data['is_ransomware']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows | Features: {X_train.shape[1]}')

### Helper: Evaluation Function

In [ ]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te, feature_names=None):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    print(f'\n=== {name} ===')
    print(classification_report(y_te, y_pred, target_names=['Benign', 'Ransomware']))

    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Benign','Ransomware'],
                yticklabels=['Benign','Ransomware'])
    plt.title(f'{name} — Confusion Matrix')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout(); plt.show()

    if feature_names is not None and hasattr(model, 'feature_importances_'):
        imp = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
        plt.figure(figsize=(10, 5))
        sns.barplot(x=imp.values, y=imp.index, palette='mako')
        plt.title(f'{name} — Feature Importance')
        plt.tight_layout(); plt.show()

    return model

### 6. Random Forest

In [ ]:
rf = evaluate('Random Forest',
              RandomForestClassifier(n_estimators=100, max_depth=16, random_state=42),
              X_train, X_test, y_train, y_test, feature_names=X.columns)

### 7. Decision Tree

In [ ]:
dt = evaluate('Decision Tree',
              DecisionTreeClassifier(max_depth=16, random_state=42),
              X_train, X_test, y_train, y_test, feature_names=X.columns)

### 8. Logistic Regression

In [ ]:
evaluate('Logistic Regression',
         LogisticRegression(max_iter=1000, random_state=42),
         X_train_sc, X_test_sc, y_train, y_test)

### 9. SVM

In [ ]:
evaluate('SVM',
         SVC(kernel='rbf', random_state=42),
         X_train_sc, X_test_sc, y_train, y_test)

### 10. KNN

In [ ]:
evaluate('KNN',
         KNeighborsClassifier(n_neighbors=5),
         X_train_sc, X_test_sc, y_train, y_test)

### 11. Naive Bayes

In [ ]:
evaluate('Naive Bayes',
         GaussianNB(),
         X_train, X_test, y_train, y_test)

### 12. Model Accuracy Comparison

In [ ]:
from sklearn.metrics import accuracy_score

models = {
    'Random Forest':      (RandomForestClassifier(n_estimators=100, max_depth=16, random_state=42), X_train, X_test),
    'Decision Tree':      (DecisionTreeClassifier(max_depth=16, random_state=42),                   X_train, X_test),
    'Logistic Regression':(LogisticRegression(max_iter=1000, random_state=42),                      X_train_sc, X_test_sc),
    'SVM':                (SVC(kernel='rbf', random_state=42),                                      X_train_sc, X_test_sc),
    'KNN':                (KNeighborsClassifier(n_neighbors=5),                                     X_train_sc, X_test_sc),
    'Naive Bayes':        (GaussianNB(),                                                             X_train, X_test),
}

results = {}
for name, (clf, Xtr, Xte) in models.items():
    clf.fit(Xtr, y_train)
    results[name] = accuracy_score(y_test, clf.predict(Xte))

res = pd.Series(results).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=res.values, y=res.index, palette='viridis')
plt.xlim(0, 1)
plt.xlabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.tight_layout()
plt.show()
print(res)

## Discussion
After evaluating multiple machine learning models for ransomware detection, **Random Forest** consistently delivers the highest accuracy and F1-score. Its ensemble nature makes it robust against overfitting and effective at capturing the complex behavioral patterns that distinguish ransomware from benign processes. Decision Tree also performs well while remaining interpretable. Logistic Regression and SVM perform reasonably on scaled features, while KNN and Naive Bayes lag slightly due to feature independence assumptions. The behavioral features — particularly `files_encrypted_per_min`, `entropy_delta`, and `shadow_copy_delete_attempt` — contribute most to detection accuracy. Future work could incorporate time-series analysis and adversarial testing.

## Reference
[1] https://github.com/MAHFUJ-7/Cybersecurity-Threat-Prediction